# 🏃‍♂️ Nextsync Sports Photo Processing Pipeline Controller

Welcome to the **Nextsync** full-stack sports photo processing pipeline. This notebook is designed to run in **Google Colab** (with GPU support if available) to process athlete photos efficiently.

### Pipeline Steps:
1. **Scans Google Drive** folder for `trigger_*.json` job files.
2. **Evaluates Quality** using OpenCV (rejects blur, dark, or bright photos).
3. **Detects & Indexes Faces** using `InsightFace` (buffalo_l model).
4. **Extracts Embeddings** and projects them to 128-dimensional vectors to match client-side face-api.js vectors.
5. **Generates Thumbnails** (800px / 400px) and uploads them to **Cloudflare R2** storage.
6. **Persists data** in the PostgreSQL database.

## 📦 Step 1: Install Dependencies

Let's install all required libraries. We will download OpenCV, InsightFace, psycopg2, and AWS SDK dependencies.

In [ ]:
# Install dependencies
!pip install -q opencv-python-headless insightface pgvector psycopg2-binary google-api-python-client google-auth-httplib2 google-auth-oauthlib numpy Pillow onnxruntime-gpu boto3

## 🔑 Step 2: Configure Environment Variables & Credentials

Input your PostgreSQL connection string, Cloudflare R2 configurations, and paths to your Google Cloud service account JSON file.

In [ ]:
import os

# 1. Database Connection URL (PostgreSQL with pgvector)
os.environ["DATABASE_URL"] = "your-database-url-here"

# 2. Cloudflare R2 Credentials (for hosting thumbnails)
os.environ["R2_ACCOUNT_ID"] = "your-r2-account-id-here"
os.environ["R2_ACCESS_KEY_ID"] = "your-r2-access-key-id-here"
os.environ["R2_SECRET_ACCESS_KEY"] = "your-r2-secret-access-key-here"
os.environ["R2_BUCKET_NAME"] = "your-r2-bucket-name-here"
os.environ["R2_PUBLIC_URL"] = "your-r2-public-url-here"

# 3. Google Drive root upload folder ID
os.environ["GOOGLE_DRIVE_FOLDER_ID"] = "your-google-drive-folder-id-here"

# Path to your Google Cloud Service Account JSON credential file
SERVICE_ACCOUNT_JSON = "service-account.json"

## 🚀 Step 3: Execute Pipeline Run

Run the pipeline driver. This parses any active trigger files and processes new photos.

In [ ]:
import sys
# Add path to import run_pipeline if it is located in a subdirectory
sys.path.append(os.getcwd())

from run_pipeline import run_pipeline

if __name__ == "__main__":
    if not os.path.exists(SERVICE_ACCOUNT_JSON):
        print(f"❌ ERROR: Please upload your google drive '{SERVICE_ACCOUNT_JSON}' file to this Colab workspace first.")
    else:
        print("🏁 Starting Nextsync pipeline runner...")
        run_pipeline(SERVICE_ACCOUNT_JSON)

## 🔄 Step 4: Run Continuous Loop (Polling mode)

This script continuously polls the Google Drive folder for any trigger files uploaded by photographers on the web interface.

In [ ]:
import time

POLL_INTERVAL_SECONDS = 20

print("📡 Nextsync Daemon: Polling Google Drive folder for trigger requests... (Press Stop to end)")
try:
    while True:
        try:
            print("Checking trigger queue...")
            run_pipeline(SERVICE_ACCOUNT_JSON)
        except Exception as e:
            print(f"Error running pipeline: {e}")
        print(f"Sleeping for {POLL_INTERVAL_SECONDS} seconds before next poll...")
        time.sleep(POLL_INTERVAL_SECONDS)
except KeyboardInterrupt:
    print("🛑 Daemon stopped by user.")